
Task

    Install necessary libraries (PEFT, datasets).
    Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer.
    Load the dataset and preprocess it for the model.
    Configure LoRA using LoraConfig.
    Apply LoRA to the pre-trained model using get_peft_model.
    Set up training arguments using TrainingArguments.
    Initialize and train the model using Trainer.
    Save the fine-tuned LoRA model.
    Load the saved LoRA model for inference using PeftModel.from_pretrained.
    Generate text using the fine-tuned model and the tokenizer.


In [1]:
!pip uninstall -y peft transformers accelerate
!pip install peft==0.10.0 transformers==4.40.0 accelerate==0.29.0

Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: accelerate 0.29.0
Uninstalling accelerate-0.29.0:
  Successfully uninstalled accelerate-0.29.0
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached accelerate-0.29.0-py3-none-any.whl.metadata (18 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)
Using cached transformers-4.40.0-py3-none-any.whl (9.0 MB)
Using cached accelerate-0.29.0-py3-none-any.whl (297 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 w

In [2]:
!mkdir cache

!pip install datasets

mkdir: cannot create directory ‘cache’: File exists


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

# Environment fix for the CUDA crash we saw earlier
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
output_directory = os.path.join(".", "peft_lab_outputs")

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto",
    pad_token_id=tokenizer.pad_token_id
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(foundation_model, lora_config)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from datasets import load_dataset

# 1. Load the dataset
quotes_dataset = load_dataset("Abirate/english_quotes")

# 2. Define the map function to tokenize the quotes
def tokenize_function(examples):
    return tokenizer(examples["quote"], truncation=True, padding="max_length", max_length=128)

# 3. Tokenize the dataset and create the 'data' variable
data = quotes_dataset["train"].map(tokenize_function, batched=True)

# 4. Final check to ensure 'data' is ready for the Trainer
print(f"Dataset 'data' defined with {len(data)} samples.")
print(f"Sample keys: {data[0].keys()}")

Dataset 'data' defined with 2508 samples.
Sample keys: dict_keys(['quote', 'author', 'tags', 'input_ids', 'attention_mask'])


In [5]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=output_directory,
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    max_steps=10,
    fp16=False,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data, # Make sure your 'data' variable cell is run before this!
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
1,3.319300
2,4.320300
3,4.876800
4,4.860500
5,4.197000
6,2.792200
7,4.206200
8,3.666300
9,3.407800
10,3.620900


TrainOutput(global_step=10, training_loss=3.926728868484497, metrics={'train_runtime': 5.2631, 'train_samples_per_second': 1.9, 'train_steps_per_second': 1.9, 'total_flos': 2327807262720.0, 'train_loss': 3.926728868484497, 'epoch': 0.003987240829346092})

In [6]:
import time
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
peft_model.to(DEVICE)

inputs = tokenizer("Two things are infinite: ", return_tensors="pt").to(DEVICE)

outputs = peft_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=30,
    do_sample=True,
    top_k=50,
    temperature=0.7,
    pad_token_id=tokenizer.pad_token_id # Explicitly set this!
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

['Two things are infinite:  number and time']
